In [8]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
import pandas as pd
import requests
from io import StringIO, BytesIO
import json
import os
import datetime

# ==============================
# 🔐 CONFIG
# ==============================

SERVICE_ACCOUNT_FILE = "D:\\Data Analytics\\PROJECTS\\Food_Data_Analysis\\fetching-data-from-drive-bc2ee8f34bde.json"

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

STAGING_FOLDER_ID = '19kHynjk43QMv6MsGR4cs6Db9hAyHo5OO'

TRACK_FILE = "processed_files.json"

# ==============================
# 🔗 CONNECT TO GOOGLE DRIVE
# ==============================

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)

service = build('drive', 'v3', credentials=credentials)

# ==============================
# 📂 LOAD TRACKING FILE
# ==============================

if os.path.exists(TRACK_FILE):
    with open(TRACK_FILE, "r") as f:
        existing_data = json.load(f)
else:
    existing_data = []

# Create set for fast lookup
processed_files = set(item["file_id"] for item in existing_data)

# ==============================
# 📂 LIST FILES
# ==============================

def list_files(service, folder_id):
    results = service.files().list(
        q=f"'{folder_id}' in parents",
        fields="files(id, name, mimeType)"
    ).execute()
    return results.get('files', [])

# ==============================
# 📥 FETCH FILES
# ==============================

files = list_files(service, STAGING_FOLDER_ID)

file_dataframes = []

for file in files:
    file_id = file['id']
    file_name = file['name']
    mime_type = file['mimeType']
    
    print(f"Processing: {file_name} | ID: {file_id}")

    # ⛔ Skip already processed files
    if file_id in processed_files:
        print(f"⏭️ Skipping already processed: {file_name}")
        continue

    # Determine download URL
    if mime_type == 'application/vnd.google-apps.spreadsheet':
        download_url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=csv"

    elif mime_type == 'text/csv':
        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"

    elif mime_type == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
        download_url = f"https://drive.google.com/uc?export=download&id={file_id}"

    else:
        print(f"Skipping unsupported file: {file_name}")
        continue

    try:
        response = requests.get(download_url)

        if response.status_code == 200:
            # Read into DataFrame
            if mime_type == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
                df = pd.read_excel(BytesIO(response.content))
            else:
                df = pd.read_csv(StringIO(response.content.decode('utf-8')))

            # ✅ Process only if not empty
            if not df.empty:
                file_dataframes.append(df)

                # ✅ Add metadata to tracking
                existing_data.append({
                    "file_id": file_id,
                    "file_name": file_name,
                    "processed_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })

                # Update set
                processed_files.add(file_id)

                # Save JSON (pretty format)
                with open(TRACK_FILE, "w") as f:
                    json.dump(existing_data, f, indent=4)

                print(f"✅ Processed & tracked: {file_name}")
            else:
                print(f"⚠️ Empty file skipped: {file_name}")

        else:
            print(f"❌ Download failed: {file_name}")

    except Exception as e:
        print(f"❌ Error processing {file_name}: {str(e)}")

# ==============================
# 🔗 COMBINE DATA
# ==============================

if file_dataframes:
    combined_df = pd.concat(file_dataframes, ignore_index=True)
    print("✅ All files combined successfully")
else:
    print("⚠️ No new files processed")

Processing: df_2021_03.csv | ID: 1GqCjV3shze6PHF3s3ug72SSBWX1ZWihR
✅ Processed & tracked: df_2021_03.csv
Processing: df_2021_02.csv | ID: 1cLclcDoe-P1KfEc1tDtPc7Lt9tX7856x
✅ Processed & tracked: df_2021_02.csv
Processing: df_2021_01.csv | ID: 1N5TAgifeG9NiYGxa-uuHKjAfAdYRPaEk
✅ Processed & tracked: df_2021_01.csv
✅ All files combined successfully


In [9]:
# ==============================
# 💾 SAVE COMBINED DATA
# ==============================

output_folder = "current_raw_data"
os.makedirs(output_folder, exist_ok=True)

# Option 1: overwrite every run (latest snapshot)
file_path = os.path.join(output_folder, "combined_data.csv")
combined_df.to_csv(file_path, index=False)

print(f"✅ Combined data saved to: {file_path}")

✅ Combined data saved to: current_raw_data\combined_data.csv


In [35]:
combined_df.shape

(120, 4)

In [36]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Report     120 non-null    object 
 1   Line Item  120 non-null    object 
 2   Date_Info  120 non-null    object 
 3   Value      120 non-null    float64
dtypes: float64(1), object(3)
memory usage: 3.9+ KB


In [37]:
combined_df.head()

,Report,Line Item,Date_Info,Value
0,SOBHA LTD Sales,Sales,3/31/2021,2109.78
1,SOBHA LTD Operating Profit,Profit,3/31/2021,675.14
2,SOBHA LTD Interest,Interest,3/31/2021,601.21
3,SOBHA LTD Profit before tax,tax,3/31/2021,75.18
4,SOBHA LTD Tax,Tax,3/31/2021,12.91


In [38]:
df = combined_df

(120, 4)

In [49]:
df = df.rename(columns={
    'Report': 'report',
    'Line Item': 'line_item',
    'Attribute': 'date_info',
    'Value': 'value'
})

In [52]:
df = df.rename(columns={
    'Report': 'report',
    'Line Item': 'line_item',
    'Date_Info': 'date_info',
    'Value': 'value'
})

In [53]:
df.columns

Index(['report', 'line_item', 'date_info', 'value'], dtype='object')

In [54]:
import mysql.connector
import pandas as pd

# Connect
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="123456",
    database="incremental_load"
)

cursor = conn.cursor()

# Ensure correct column order
df = df[['report', 'line_item', 'date_info', 'value']]

# Handle NULLs
df = df.where(pd.notnull(df), None)

# Convert to list of tuples
data = [tuple(row) for row in df.to_numpy()]

# Insert query
query = """
INSERT INTO competitors_data (report, line_item, date_info, value)
VALUES (%s, %s, %s, %s)
"""

# 🔥 Batch insert (FAST)
batch_size = 1000

for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    cursor.executemany(query, batch)
    conn.commit()

print("Data inserted fast 🚀")

cursor.close()
conn.close()

Data inserted fast 🚀


In [ ]:
print(df.columns)

Index(['Report', 'Line Item', 'Attribute', 'Value'], dtype='object')
